In [1]:
import ctypes
import numpy as np
from pathlib import Path

lib = ctypes.CDLL("./rbf_lib.so")

lib.entrainer.argtypes = [
    ctypes.POINTER(ctypes.c_double),  # X
    ctypes.POINTER(ctypes.c_int),     # y
    ctypes.c_int,                     # n
    ctypes.c_int,                     # d
    ctypes.c_int,                     # nb_centres
    ctypes.c_double,                  # gamma
    ctypes.c_uint,                    # seed
    ctypes.POINTER(ctypes.c_double),  # centres_out
    ctypes.POINTER(ctypes.c_double),  # W_out
]
lib.entrainer.restype = None

lib.entrainer_rosenblatt.argtypes = [
    ctypes.POINTER(ctypes.c_double),  # X
    ctypes.POINTER(ctypes.c_int),     # y
    ctypes.c_int,                     # n
    ctypes.c_int,                     # d
    ctypes.c_int,                     # nb_centres
    ctypes.c_double,                  # gamma
    ctypes.c_uint,                    # seed
    ctypes.c_int,                     # epochs
    ctypes.c_double,                  # lr
    ctypes.POINTER(ctypes.c_double),  # centres_out
    ctypes.POINTER(ctypes.c_double),  # W_out
    ctypes.POINTER(ctypes.c_double),  # b_out
]
lib.entrainer_rosenblatt.restype = None

lib.predire_batch.argtypes = [
    ctypes.POINTER(ctypes.c_double),  # X_test
    ctypes.c_int,                     # n_test
    ctypes.c_int,                     # d
    ctypes.POINTER(ctypes.c_double),  # centres_plat
    ctypes.c_int,                     # nb_centres
    ctypes.POINTER(ctypes.c_double),  # W_plat
    ctypes.POINTER(ctypes.c_double),  # b_plat
    ctypes.c_double,                  # gamma
    ctypes.POINTER(ctypes.c_int),     # predictions_out
]
lib.predire_batch.restype = None

In [2]:
def lire_X(chemin):
    with open(chemin, "rb") as f:
        n = int(np.fromfile(f, dtype=np.int32, count=1)[0])
        d = int(np.fromfile(f, dtype=np.int32, count=1)[0])
        X = np.fromfile(f, dtype=np.float32, count=n * d)
    return np.ascontiguousarray(X.reshape(n, d).astype(np.float64)), n, d

def lire_y(chemin):
    with open(chemin, "rb") as f:
        n = int(np.fromfile(f, dtype=np.int32, count=1)[0])
        y = np.fromfile(f, dtype=np.int32, count=n)
    return np.ascontiguousarray(y), n

racine = Path.cwd()
while racine != racine.parent and not (racine / "preprocessing").exists():
    racine = racine.parent
base = racine / "datasets" / "transformed" / "nb" / "normalisee"

X_train, n_train, d = lire_X(base / "X_train.f32bin")
y_train, _ = lire_y(base / "y_train.i32bin")
X_test, n_test, _ = lire_X(base / "X_test.f32bin")
y_test, _ = lire_y(base / "y_test.i32bin")
print(n_train, n_test, d)

1200 300 16384


In [3]:
gamma = 0.01
nb_centres = 100
seed = 42
K_CLASSES = 3

centres = np.zeros(nb_centres * d, dtype=np.float64)
W = np.zeros(K_CLASSES * nb_centres, dtype=np.float64)
b = np.zeros(K_CLASSES, dtype=np.float64)   # nul en mode pinv

pd = ctypes.POINTER(ctypes.c_double)
pi = ctypes.POINTER(ctypes.c_int)

lib.entrainer(
    X_train.ctypes.data_as(pd), y_train.ctypes.data_as(pi),
    n_train, d, nb_centres, gamma, seed,
    centres.ctypes.data_as(pd), W.ctypes.data_as(pd),
)

predictions = np.zeros(n_test, dtype=np.int32)
lib.predire_batch(
    X_test.ctypes.data_as(pd), n_test, d,
    centres.ctypes.data_as(pd), nb_centres,
    W.ctypes.data_as(pd), b.ctypes.data_as(pd), gamma,
    predictions.ctypes.data_as(pi),
)

acc_test = (predictions == y_test).mean()
print("acc test (lib pinv) :", round(acc_test, 3))

acc test (lib pinv) : 0.51
